# 4.3 Lab: Smart KV Caching - Attention-Aware Eviction[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.3_smart_kv_caching/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/04_kv_cache_engineering/04.3_smart_kv_caching/lab.ipynb)This lab loads Mistral-7B, extracts real attention patterns, and demonstrates:1. The power-law distribution of attention (which tokens matter)2. H2O-style cumulative score eviction vs random eviction3. SnapKV-style one-shot compression quality4. StreamingLLM sink behavior5. Memory savings measurement

In [ ]:
# --- Setup: install dependencies and detect device ---import subprocess, syssubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch", "matplotlib", "numpy"])# PyTorch: GPU operations and attention weight extractionimport torch# NumPy: array operations for attention analysisimport numpy as np# Matplotlib: attention distribution visualizationsimport matplotlib.pyplot as plt# HuggingFace: load Mistral-7B with output_attentions=Truefrom transformers import AutoModelForCausalLM, AutoTokenizer# Device detection# Use GPU if available, CPU fallback for testingDEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"Device: {DEVICE}")if DEVICE == "cuda":    # Print GPU memory info    # Report GPU VRAM for experiment context    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB)")

In [ ]:
# --- Load Mistral-7B and tokenize a long prompt ---# Non-gated model: no auth token requiredMODEL_NAME = "mistralai/Mistral-7B-v0.1"# Load in float16 for GPU efficiency# Load tokenizer for text-to-token conversionmistral_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)# Load model with attention output enabled for analysismistral_model = AutoModelForCausalLM.from_pretrained(    MODEL_NAME,    torch_dtype=torch.float16,    device_map="auto",    output_attentions=True,  # Required for attention extraction)mistral_model.eval()# Long prompt to demonstrate attention patterns across many positions# Long prompt covering transformer fundamentals to generate diverse attention patternslong_prompt = """The transformer architecture processes input sequences through multiple layers of self-attention and feed-forward networks. Each attention head computes query, key, and value projections, then uses softmax over the dot products of queries and keys to weight the values. In practice, most attention weight concentrates on a small subset of positions. The first few tokens in any sequence attract disproportionate attention regardless of their content, a phenomenon called attention sinks. Content-bearing tokens like nouns and verbs also attract high attention from subsequent positions that reference their meaning. This concentration follows a power law: the top 5 percent of positions typically capture 60 to 80 percent of the total attention mass. Understanding this distribution is the key insight behind smart KV caching strategies that achieve dramatic memory savings with minimal quality loss."""# Tokenize# Tokenize and move to GPU for forward passprompt_token_ids = mistral_tokenizer(long_prompt, return_tensors="pt").input_ids.to(DEVICE)seq_length = prompt_token_ids.shape[1]print(f"Prompt length: {seq_length} tokens")

## Experiment 1: Power-Law Attention DistributionWe extract attention weights from a forward pass and verify that attention is highly concentrated on a few positions.

In [ ]:
# --- Extract attention weights from full forward pass ---with torch.no_grad():    model_outputs = mistral_model(prompt_token_ids, output_attentions=True)# attention_weights_all_layers: list of [1, num_heads, seq_len, seq_len] per layer# List of attention tensors: one [1, heads, seq, seq] per layerattention_weights_all_layers = model_outputs.attentionsnum_layers_model = len(attention_weights_all_layers)num_heads_model = attention_weights_all_layers[0].shape[1]print(f"Layers: {num_layers_model}, Heads per layer: {num_heads_model}, Seq length: {seq_length}")# Compute per-position importance: average attention received across all queries, heads, layers# For each position j, sum attention[i][j] for all i > j (causal), then average# Accumulate attention received by each position across all layersposition_importance_scores = torch.zeros(seq_length, device=DEVICE)for layer_attn_weights in attention_weights_all_layers:    # layer_attn_weights: [1, heads, seq, seq]    # Sum attention received by each position (column sum of lower triangle)    col_sums = layer_attn_weights[0].sum(dim=1).sum(dim=0)  # [seq_len] - sum over heads then queries    position_importance_scores += col_sums# Normalize to get fraction of total attention per position# Normalize to get fraction of total attention per positionposition_importance_normalized = position_importance_scores / position_importance_scores.sum()position_importance_np = position_importance_normalized.cpu().numpy()# Sort and compute cumulative distribution# Sort descending for cumulative distribution computationsorted_importance_vals = np.sort(position_importance_np)[::-1]  # descending# Cumulative sum reveals concentration (power-law shape)cumulative_attention_fraction = np.cumsum(sorted_importance_vals)# Report concentration# Compute top-5% and top-10% thresholds for reportingtop5_pct_count = max(1, int(0.05 * seq_length))top10_pct_count = max(1, int(0.10 * seq_length))print(f"Top 5% of positions ({top5_pct_count} tokens) capture {cumulative_attention_fraction[top5_pct_count-1]*100:.1f}% of attention")print(f"Top 10% of positions ({top10_pct_count} tokens) capture {cumulative_attention_fraction[top10_pct_count-1]*100:.1f}% of attention")

In [ ]:
# --- Plot: Attention concentration curve ---fig_concentration, ax_concentration = plt.subplots(1, 2, figsize=(12, 4))# Left: cumulative attention vs fraction of positionsposition_fractions = np.arange(1, seq_length + 1) / seq_length# Plot line for this metricax_concentration[0].plot(position_fractions * 100, cumulative_attention_fraction * 100, linewidth=2, color="#2563eb")# Reference line for comparison thresholdax_concentration[0].axhline(y=80, color="#991b1b", linestyle="--", alpha=0.7, label="80% attention")# Reference line for comparison thresholdax_concentration[0].axhline(y=95, color="#166534", linestyle="--", alpha=0.7, label="95% attention")# Configure axis propertyax_concentration[0].set_xlabel("Fraction of positions (sorted by importance) [%]")# Configure axis propertyax_concentration[0].set_ylabel("Cumulative attention captured [%]")# Configure axis propertyax_concentration[0].set_title("Power Law: Few Tokens Capture Most Attention")# Execute operationax_concentration[0].legend()# Execute operationax_concentration[0].grid(True, alpha=0.3)# Right: per-position importance (raw order)ax_concentration[1].bar(range(seq_length), position_importance_np, color="#2563eb", alpha=0.7, width=1.0)# Configure axis propertyax_concentration[1].set_xlabel("Token position")# Configure axis propertyax_concentration[1].set_ylabel("Fraction of total attention received")# Configure axis propertyax_concentration[1].set_title("Per-Position Importance (Attention Sinks Visible)")# Execute operationax_concentration[1].grid(True, alpha=0.3)# Adjust spacing to prevent label overlapplt.tight_layout()# Render the chartplt.show()

## Experiment 2: H2O vs Random EvictionWe simulate H2O (keep highest cumulative-score tokens) vs random eviction at various budgets, measuring how much attention signal is retained.

In [ ]:
# --- H2O vs Random eviction: attention retention at various budgets ---# Test budgets from 10% to 75% of original sequence lengthbudget_fractions_list = [0.1, 0.2, 0.3, 0.5, 0.75]# H2O: keep highest cumulative-score tokens (oracle eviction)h2o_retention_results = []# Random baseline: average over 50 trials for stabilityrandom_retention_results = []# Iterate over each item in the collectionfor budget_frac in budget_fractions_list:    # Process this step    budget_k = max(1, int(budget_frac * seq_length))        # H2O: keep top-k by cumulative score (our position_importance_scores)    topk_indices_h2o = torch.topk(position_importance_scores, budget_k).indices    # Aggregate values across the dimension    h2o_retained_signal = position_importance_normalized[topk_indices_h2o].sum().item()    # Record this measurement    h2o_retention_results.append(h2o_retained_signal * 100)        # Random: average over 50 trials    random_trials_signals = []    # Iterate over each item in the collection    for _ in range(50):        # Create tensor for computation        random_indices_trial = torch.randperm(seq_length, device=DEVICE)[:budget_k]        # Aggregate values across the dimension        random_signal_trial = position_importance_normalized[random_indices_trial].sum().item()        # Record this measurement        random_trials_signals.append(random_signal_trial * 100)    # Compute statistical summary of results    random_retention_results.append(np.mean(random_trials_signals))# Plot comparisonfig_eviction, ax_eviction = plt.subplots(figsize=(8, 5))# Iterate over each item in the collectionbudget_pcts = [f * 100 for f in budget_fractions_list]# Plot line for this metricax_eviction.plot(budget_pcts, h2o_retention_results, "o-", color="#2563eb", linewidth=2, markersize=8, label="H2O (attention-guided)")# Plot line for this metricax_eviction.plot(budget_pcts, random_retention_results, "s--", color="#991b1b", linewidth=2, markersize=8, label="Random eviction")ax_eviction.set_xlabel("Cache budget [% of positions retained]")ax_eviction.set_ylabel("Attention signal retained [%]")ax_eviction.set_title("H2O vs Random Eviction: Signal Retention")ax_eviction.legend()ax_eviction.grid(True, alpha=0.3)# Process this stepax_eviction.set_ylim(0, 105)# Adjust spacing to prevent label overlapplt.tight_layout()plt.show()# Print tableprint(f"{'Budget':>8} {'H2O':>10} {'Random':>10} {'Gap':>8}")for i, bfrac in enumerate(budget_fractions_list):    gap_val = h2o_retention_results[i] - random_retention_results[i]    print(f"{bfrac*100:>6.0f}% {h2o_retention_results[i]:>9.1f}% {random_retention_results[i]:>9.1f}% {gap_val:>7.1f}%")

## Experiment 3: SnapKV Simulation - Observation Window EffectivenessWe simulate SnapKV by scoring positions using only the last N tokens (observation window) and checking how well this predicts full-sequence importance.

In [ ]:
# --- SnapKV: observation window predicts full-sequence importance ---# Kendall tau: rank correlation between window-based and full importancefrom scipy.stats import kendalltau# Use the last layer attention as our reference (deep layers = sharper attention)# Use deepest layer (sharpest attention) as referencelast_layer_attn_tensor = attention_weights_all_layers[-1][0]  # [heads, seq, seq]# Full-sequence importance: average attention received across all heads and all query positions# Ground truth: importance computed from ALL query positionsfull_seq_importance_per_pos = last_layer_attn_tensor.sum(dim=0).sum(dim=0)  # [seq_len]# Observation windows of different sizes# SnapKV paper tests observation windows of 8-64 tokensobs_window_sizes = [8, 16, 32, 64]# Collect rank correlation for each window sizekendall_tau_results = []# Iterate over each item in the collectionfor obs_win_size in obs_window_sizes:    # SnapKV scoring: importance from last obs_win_size query positions only    obs_window_start_idx = max(0, seq_length - obs_win_size)    # Aggregate values across the dimension    obs_importance_per_pos = last_layer_attn_tensor[:, obs_window_start_idx:, :].sum(dim=0).sum(dim=0)        # Kendall tau rank correlation with full-sequence importance    tau_val, _ = kendalltau(        # Move to CPU and convert to numpy for plotting        full_seq_importance_per_pos.cpu().numpy(),        # Move to CPU and convert to numpy for plotting        obs_importance_per_pos.cpu().numpy()    # Process this step    )    # Record this measurement    kendall_tau_results.append(tau_val)    # Display formatted result    print(f"Observation window = {obs_win_size:>3} tokens -> Kendall tau = {tau_val:.3f}")# Plotfig_snapkv, ax_snapkv = plt.subplots(figsize=(7, 4))ax_snapkv.bar(range(len(obs_window_sizes)), kendall_tau_results, color="#166534", alpha=0.8)ax_snapkv.set_xticks(range(len(obs_window_sizes)))ax_snapkv.set_xticklabels([f"{w} tokens" for w in obs_window_sizes])ax_snapkv.set_ylabel("Kendall Tau (rank correlation)")ax_snapkv.set_title("SnapKV: Observation Window vs Full-Sequence Importance Correlation")ax_snapkv.set_ylim(0, 1.0)ax_snapkv.axhline(y=0.85, color="#991b1b", linestyle="--", alpha=0.7, label="0.85 threshold (Li et al. 2024)")ax_snapkv.legend()ax_snapkv.grid(True, alpha=0.3, axis="y")plt.tight_layout()plt.show()

## Experiment 4: Attention Sinks - StreamingLLM ValidationWe verify the attention sink phenomenon: first few tokens attract disproportionate attention regardless of content.

In [ ]:
# --- Attention sinks: first tokens attract outsized attention ---# Average attention received by each position across ALL layers and heads# Use normalized importance computed earlier (across all layers/heads)avg_attn_per_position = position_importance_np# Compare first 4 tokens vs middle tokens vs last tokens# Attention sinks: first 4 tokens (BOS + initial tokens)sink_positions_attn = avg_attn_per_position[:4]# Middle tokens: bulk of the sequence (baseline reference)middle_positions_attn = avg_attn_per_position[4:seq_length-32]# Recent tokens: recency bias in causal attentionrecent_positions_attn = avg_attn_per_position[seq_length-32:]# Aggregate values across the dimensionsink_avg_score = sink_positions_attn.mean()# Aggregate values across the dimensionmiddle_avg_score = middle_positions_attn.mean()# Aggregate values across the dimensionrecent_avg_score = recent_positions_attn.mean()# Display formatted resultprint(f"Average importance per token:")# Display formatted resultprint(f"  Attention sinks (pos 0-3):    {sink_avg_score:.5f} ({sink_avg_score/middle_avg_score:.1f}x vs middle)")# Display formatted resultprint(f"  Middle tokens (pos 4 to -32): {middle_avg_score:.5f} (baseline)")# Display formatted resultprint(f"  Recent tokens (last 32):      {recent_avg_score:.5f} ({recent_avg_score/middle_avg_score:.1f}x vs middle)")# Plot: zoom into first 20 positions to show sink effectfig_sinks, ax_sinks = plt.subplots(figsize=(10, 4))# Process this stepfirst_n_positions = min(40, seq_length)ax_sinks.bar(range(first_n_positions), avg_attn_per_position[:first_n_positions], color="#2563eb", alpha=0.7)# Highlight sinksax_sinks.bar(range(4), avg_attn_per_position[:4], color="#991b1b", alpha=0.9, label="Attention sinks (pos 0-3)")ax_sinks.axhline(y=middle_avg_score, color="#64748b", linestyle="--", label=f"Middle average ({middle_avg_score:.5f})")ax_sinks.set_xlabel("Token position")ax_sinks.set_ylabel("Importance (fraction of total attention)")ax_sinks.set_title("Attention Sinks: First Tokens Attract Disproportionate Attention")ax_sinks.legend()ax_sinks.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Experiment 5: Memory Savings CalculationConcrete memory math for Mistral-7B with different smart caching strategies.

In [ ]:
# --- Memory savings calculation for Mistral-7B ---# Mistral-7B: 32 layers, 8 KV heads (GQA), 128 head_dim, FP16# Mistral-7B architecture constants for memory calculationNUM_LAYERS_MISTRAL = 32# Configuration parameterNUM_KV_HEADS_MISTRAL = 8  # GQA: 8 KV heads (not 32)# Configuration parameterHEAD_DIM_MISTRAL = 128# Configuration parameterBYTES_PER_ELEMENT = 2  # FP16# Memory per token per layer: 2 (K+V) * num_kv_heads * head_dim * bytes# Memory per token per layer: 2*(K+V) * kv_heads * head_dim * dtype_bytesbytes_per_token_per_layer = 2 * NUM_KV_HEADS_MISTRAL * HEAD_DIM_MISTRAL * BYTES_PER_ELEMENT# Total across all layers# Sum across all 32 layers for total per-token KV costbytes_per_token_total = bytes_per_token_per_layer * NUM_LAYERS_MISTRAL# Process this stepkb_per_token = bytes_per_token_total / 1024# Display formatted resultprint(f"Mistral-7B KV cache cost per token: {bytes_per_token_total:,} bytes = {kb_per_token:.1f} KB")# Process this stepprint()# Scenarios# Test at typical serving lengths from 4K to 128Kcontext_lengths_list = [4096, 8192, 32768, 131072]# Caching strategies: fraction of tokens retained in KV cachestrategies_config = {    # Process this step    "Full cache": 1.0,    # Process this step    "SnapKV (4x)": 0.25,    # Process this step    "H2O (5x)": 0.20,    # Process this step    "StreamingLLM (S=4, W=2048)": None,  # Fixed at 2052 tokens# Process this step}# Display formatted resultprint(f"{'Context':>10} {'Full Cache':>12} {'SnapKV 4x':>12} {'H2O 5x':>12} {'StreamingLLM':>14}")# Process this stepprint("-" * 65)# Iterate over each item in the collectionmemory_data_for_plot = {s: [] for s in strategies_config}# Iterate over each item in the collectionfor ctx_len in context_lengths_list:    # Initialize results collection    row_values = []    # Iterate over each item in the collection    for strategy_name, ratio_val in strategies_config.items():        # Conditional check        if ratio_val is None:            # StreamingLLM: fixed 2052 tokens            effective_tokens = 2052        # Process this step        else:            effective_tokens = int(ctx_len * ratio_val)        mem_gb = (effective_tokens * bytes_per_token_total) / (1024**3)        memory_data_for_plot[strategy_name].append(mem_gb)        row_values.append(f"{mem_gb:.2f} GB")    print(f"{ctx_len:>10,} {row_values[0]:>12} {row_values[1]:>12} {row_values[2]:>12} {row_values[3]:>14}")# Plot memory comparisonfig_mem, ax_mem = plt.subplots(figsize=(9, 5))colors_map = {"Full cache": "#991b1b", "SnapKV (4x)": "#166534", "H2O (5x)": "#2563eb", "StreamingLLM (S=4, W=2048)": "#7c3aed"}for strategy_name, mem_values in memory_data_for_plot.items():    ax_mem.plot(context_lengths_list, mem_values, "o-", label=strategy_name, color=colors_map[strategy_name], linewidth=2)ax_mem.set_xlabel("Context length (tokens)")ax_mem.set_ylabel("KV cache memory per request (GB)")ax_mem.set_title("Mistral-7B: KV Cache Memory with Smart Caching")ax_mem.set_xscale("log", base=2)ax_mem.legend()ax_mem.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Experiment 6: Layer-Wise Attention EntropyDifferent layers have different attention concentration. Deep layers concentrate more sharply and tolerate aggressive eviction.

In [ ]:
# --- Layer-wise attention entropy (validates adaptive budget allocation) ---# Compute entropy per layer to validate adaptive budget allocationlayer_entropies_list = []# Iterate over each item in the collectionfor layer_idx, layer_attn_tensor in enumerate(attention_weights_all_layers):    # layer_attn_tensor: [1, heads, seq, seq]    attn_probs = layer_attn_tensor[0]  # [heads, seq, seq]    # Compute entropy: -sum(p * log(p)) averaged over heads and query positions    log_probs = torch.log(attn_probs + 1e-10)    # Aggregate values across the dimension    entropy_per_query = -(attn_probs * log_probs).sum(dim=-1)  # [heads, seq]    # Aggregate values across the dimension    mean_entropy_val = entropy_per_query.mean().item()    # Record this measurement    layer_entropies_list.append(mean_entropy_val)# Plotfig_entropy, ax_entropy = plt.subplots(figsize=(10, 4))# Color by entropy: light blue = high entropy, dark blue = lowlayer_colors = ["#dbeafe" if e > np.median(layer_entropies_list) else "#2563eb" for e in layer_entropies_list]# Draw bar chart for this metricax_entropy.bar(range(num_layers_model), layer_entropies_list, color=layer_colors, edgecolor="#000", linewidth=0.5)# Reference line for comparison thresholdax_entropy.axhline(y=np.median(layer_entropies_list), color="#991b1b", linestyle="--", label=f"Median entropy ({np.median(layer_entropies_list):.2f})")ax_entropy.set_xlabel("Layer index")ax_entropy.set_ylabel("Mean attention entropy (nats)")ax_entropy.set_title("Layer-Wise Entropy: Deep Layers Concentrate More (Tolerate Aggressive Eviction)")ax_entropy.legend()ax_entropy.grid(True, alpha=0.3, axis="y")# Adjust spacing to prevent label overlapplt.tight_layout()# Render the chartplt.show()# Summary# Shallow layers have diffuse attention (need larger KV budget)shallow_entropy_avg = np.mean(layer_entropies_list[:8])# Deep layers concentrate sharply (tolerate aggressive eviction)deep_entropy_avg = np.mean(layer_entropies_list[-8:])print(f"Shallow layers (0-7) avg entropy: {shallow_entropy_avg:.3f} (diffuse attention, need larger budget)")print(f"Deep layers ({num_layers_model-8}-{num_layers_model-1}) avg entropy: {deep_entropy_avg:.3f} (concentrated, tolerate small budget)")print(f"Ratio: shallow/deep = {shallow_entropy_avg/deep_entropy_avg:.2f}x")

## Key Takeaways1. **Attention follows a power law** - top 5-10% of tokens capture 60-95% of attention signal2. **H2O retains 95%+ signal at 20% budget** vs random eviction retaining only proportional signal3. **SnapKV observation windows work** - last 32-64 tokens predict full-sequence importance with tau > 0.854. **Attention sinks are real** - first 4 tokens attract 5-10x more attention than middle tokens5. **Memory savings are dramatic** - 4x compression on 32K context saves ~3 GB per request on Mistral-7B6. **Deep layers can be compressed more** - their entropy is lower, meaning attention is already concentrated